# Bonus 1 : de 10 000 à 100 millions de lignes

**Pour ceux qui veulent voir du code.** À l'exercice 1, on a retiré le comptage au LLM pour le confier à DuckDB. Ici, on regarde ce qui se passe quand **le volume** augmente, sans LLM du tout.

**La question (toujours la même) :** dans une journée de journaux d'authentification, quelles adresses IP accumulent le plus d'**échecs de connexion**, et sur combien de comptes différents ? (signature d'une force brute ou d'un *password spraying*)

On pose cette question sur 10 000, 1 million, 10 millions puis **100 millions** de lignes :

| | Avant | Après |
|---|---|---|
| Format | CSV (texte, ligne par ligne) | **Parquet** (colonnes, typé, compressé) |
| Moteur | pandas (tout en mémoire, exécution immédiate) | **Polars** (Rust) ou **DuckDB** (C++) : exécution paresseuse, en flux, multi-cœurs |

Pas besoin de GPU : *Exécution > Modifier le type d'exécution > CPU*.

In [ ]:
# @title Installation et outils de mesure (cliquer pour voir le code)
!pip install -q polars duckdb psutil

import os, time, threading
import duckdb, psutil
import pandas as pd
import polars as pl

RESULTATS = []  # toutes les mesures de la séance


class mesurer:
    """Mesure la durée et le pic de mémoire (RAM) d'un bloc de code."""
    def __init__(self, methode, n):
        self.methode, self.n = methode, n

    def __enter__(self):
        self.proc = psutil.Process()
        self.base = self.pic = self.proc.memory_info().rss
        self.actif = True

        def surveiller():
            while self.actif:
                self.pic = max(self.pic, self.proc.memory_info().rss)
                time.sleep(0.02)

        self.fil = threading.Thread(target=surveiller)
        self.fil.start()
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        duree = time.perf_counter() - self.t0
        self.actif = False
        self.fil.join()
        memoire = (self.pic - self.base) / 1e6
        RESULTATS.append({"méthode": self.methode, "lignes": milliers(self.n),
                          "durée (s)": round(duree, 2), "pic RAM (Mo)": round(memoire)})
        print(f"{self.methode} sur {milliers(self.n)} lignes : {duree:.2f} s, pic de RAM ≈ {milliers(memoire)} Mo")


def milliers(x):
    return f"{x:,.0f}".replace(",", " ")


def taille(fichier):
    return milliers(os.path.getsize(fichier) / 1e6) + " Mo"


In [ ]:
# @title Générateur de journaux d'authentification (cliquer pour voir le code)
def generer(n, fichier):
    """Écrit n connexions dans un fichier CSV ou Parquet (selon l'extension).
    1 % du trafic vient de 20 IP d'attaque qui échouent presque toujours."""
    duckdb.sql(f"""
    COPY (
      SELECT
        TIMESTAMP '2026-10-26' + to_microseconds((i * 86400000000 // {n})::BIGINT) AS horodatage,
        CASE WHEN attaque THEN '185.220.101.' || (random() * 20)::INT
             ELSE '10.0.' || (random() * 255)::INT || '.' || (random() * 255)::INT
        END AS ip_src,
        'user' || (random() * 5000)::INT AS utilisateur,
        ['ssh', 'rdp', 'vpn', 'web'][1 + (random() * 3)::INT] AS service,
        CASE WHEN attaque THEN random() < 0.02 ELSE random() < 0.9 END AS succes
      FROM (SELECT i, random() < 0.01 AS attaque FROM range({n}) t(i))
    ) TO '{fichier}'""")

## Avant : pandas sur un CSV

Le code qu'on écrit tous spontanément, et qui marche très bien… à 10 000 lignes.

In [ ]:
generer(10_000, "logs_10k.csv")
pd.read_csv("logs_10k.csv").head()

In [ ]:
def analyse_pandas(fichier_csv):
    df = pd.read_csv(fichier_csv)            # 1. tout le fichier est chargé en mémoire
    echecs = df[~df["succes"]]               # 2. une copie filtrée
    return (echecs.groupby("ip_src")         # 3. agrégation
            .agg(n_echecs=("utilisateur", "size"), n_comptes=("utilisateur", "nunique"))
            .sort_values("n_echecs", ascending=False)
            .head(10))


with mesurer("pandas + CSV", 10_000):
    top = analyse_pandas("logs_10k.csv")
top

### Le même code, 100 fois puis 1 000 fois plus de données

In [ ]:
for n, f in [(1_000_000, "logs_1M.csv"), (10_000_000, "logs_10M.csv")]:
    generer(n, f)
    print(f, ":", taille(f))
    with mesurer("pandas + CSV", n):
        analyse_pandas(f)

In [ ]:
# Et à 100 millions ? On extrapole au lieu de lancer : la session Colab planterait.
pic_10M = RESULTATS[-1]["pic RAM (Mo)"]
ram_colab = psutil.virtual_memory().total / 1e6
print(f"Pic de RAM à 10 M de lignes : {milliers(pic_10M)} Mo")
print(f"Estimation à 100 M de lignes : ≈ {milliers(10 * pic_10M)} Mo")
print(f"RAM totale de cette machine  : {milliers(ram_colab)} Mo")

## À faire
1. Comment évoluent la durée et la mémoire quand on multiplie les données par 10 ?
2. Combien de colonnes du fichier sont vraiment utiles pour répondre à la question ? Combien pandas en charge ?
3. Combien de cœurs a cette machine (`os.cpu_count()`) ? Combien pandas en utilise-t-il ?

## Après, étape 1 : changer de format (CSV → Parquet)

Un CSV est du texte : pour lire une colonne, il faut lire toutes les lignes en entier, puis deviner les types. **Parquet** range les données **par colonne**, typées et compressées, avec des statistiques (min, max) par bloc. On peut donc lire uniquement les colonnes utiles et sauter les blocs inutiles.

In [ ]:
for n, f in [(10_000_000, "logs_10M.parquet")]:
    generer(n, f)

print("CSV     :", taille("logs_10M.csv"))
print("Parquet :", taille("logs_10M.parquet"))

with mesurer("pandas + Parquet (3 colonnes)", 10_000_000):
    df = pd.read_parquet("logs_10M.parquet", columns=["ip_src", "utilisateur", "succes"])
del df

## Après, étape 2 : changer de moteur (Polars, écrit en Rust)

Presque le même code, avec deux différences de fond :
- `scan_parquet` ne lit rien : il construit un **plan de requête** (exécution paresseuse, *lazy*).
- `collect(engine="streaming")` exécute ce plan **par morceaux**, sur tous les cœurs, sans jamais charger tout le fichier.

In [ ]:
def analyse_polars(fichier_parquet):
    return (pl.scan_parquet(fichier_parquet)
            .filter(~pl.col("succes"))
            .group_by("ip_src")
            .agg(n_echecs=pl.len(), n_comptes=pl.col("utilisateur").n_unique())
            .sort("n_echecs", descending=True)
            .head(10)
            .collect(engine="streaming"))


with mesurer("Polars + Parquet", 10_000_000):
    top = analyse_polars("logs_10M.parquet")
top

In [ ]:
# Le plan optimisé : Polars ne lit que les colonnes utiles (PROJECT) et filtre dès la lecture (SELECTION)
print(pl.scan_parquet("logs_10M.parquet")
        .filter(~pl.col("succes"))
        .group_by("ip_src").agg(pl.len())
        .explain())

## Après, étape 3 : une requête SQL (DuckDB)

La même question en SQL, directement sur le fichier. C'est exactement ce que font `stats count by ip_src` dans Splunk ou `summarize count() by ip_src` dans Sentinel : **le SIEM fait déjà ce travail pour vous**.

In [ ]:
def analyse_duckdb(fichier_parquet):
    return duckdb.sql(f"""
        SELECT ip_src, count(*) AS n_echecs, count(DISTINCT utilisateur) AS n_comptes
        FROM '{fichier_parquet}'
        WHERE NOT succes
        GROUP BY ip_src
        ORDER BY n_echecs DESC
        LIMIT 10""").df()


with mesurer("DuckDB + Parquet", 10_000_000):
    top = analyse_duckdb("logs_10M.parquet")
top

## 100 millions de lignes

On génère directement en Parquet (≈ 1,3 Go ; 2 à 4 minutes sur Colab), puis on relance Polars et DuckDB. On ne lance pas pandas.

In [ ]:
N = 100_000_000      # baissez à 50_000_000 si la génération est trop longue
generer(N, "logs_100M.parquet")
print("Parquet :", taille("logs_100M.parquet"))

with mesurer("Polars + Parquet", N):
    top = analyse_polars("logs_100M.parquet")
with mesurer("DuckDB + Parquet", N):
    top = analyse_duckdb("logs_100M.parquet")
top

In [ ]:
# Bilan de toutes les mesures
pd.DataFrame(RESULTATS)

## Ce qu'il faut retenir

| Ce qui a changé | Pourquoi ça compte |
|---|---|
| **CSV → Parquet** | 4 à 5 fois plus petit ; on ne lit que les colonnes et les blocs utiles |
| **Exécution immédiate → paresseuse** | le moteur voit toute la requête et l'optimise avant de lire quoi que ce soit |
| **Tout en mémoire → en flux** | la RAM ne dépend plus de la taille du fichier, mais du nombre de groupes |
| **Python, 1 cœur → Rust / C++, tous les cœurs** | calcul vectorisé et parallèle |

**Et au-delà ?** Une seule machine bien utilisée va loin (des centaines de Go). Au-delà, on partitionne les fichiers (un Parquet par jour) et on passe à des moteurs distribués (Spark, ClickHouse, BigQuery)… ou on laisse le SIEM le faire. Le message du chapitre ne change pas : **savoir à qui confier quoi**.

## À faire
1. Ajoutez à la requête DuckDB un filtre sur le service `ssh` et une fenêtre d'une heure (`date_trunc('hour', horodatage)`). Le temps change-t-il beaucoup ?
2. Réécrivez `analyse_pandas` pour lire le Parquet avec `columns=[...]`. Jusqu'à combien de lignes tient-elle sur Colab ?
3. Pourquoi Polars utilise-t-il plus de mémoire que DuckDB à 100 M de lignes ? (indice : `n_unique`, et le nombre d'IP distinctes)